# 01 - EDA (Phân tích khám phá dữ liệu)### Đề tài: Phân loại nấm ăn được hay có độcNotebook này đọc trực tiếp `dataset.zip` trong `ai-models/data/`, khảo sát và trực quan hoá dữ liệu.**Cách chạy:** Runtime → Restart & Run all (chạy lại từ đầu để đảm bảo không lỗi thứ tự cell).

In [ ]:
# Cài thư viện (Colab thường đã có sẵn, chạy để chắc chắn)!pip -q install pandas matplotlib seaborn scipy

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom itertools import combinationsfrom scipy.stats import chi2_contingencysns.set_theme(style="whitegrid")pd.set_option("display.max_columns", None)

## 1. Đọc dữ liệu từ dataset.zipNếu chạy trên Colab: upload `dataset.zip` (từ `ai-models/data/`) vào cùng thư mục, hoặc mount Google Drive rồi chỉnh đường dẫn.

In [ ]:
import zipfileDATA_ZIP = "dataset.zip"   # đổi đường dẫn nếu để ở chỗ khác, ví dụ "/content/drive/MyDrive/.../dataset.zip"with zipfile.ZipFile(DATA_ZIP) as z:    print(z.namelist())    with z.open("mushrooms.csv") as f:        df = pd.read_csv(f)df["class_full"] = df["class"].map({"e": "edible (ăn được)", "p": "poisonous (độc)"})print(df.shape)df.head()

## 2. Kiểm tra tổng quan- Kiểu dữ liệu của từng cột- Giá trị thiếu (ký hiệu `?`)- Số lượng giá trị duy nhất mỗi cột

In [ ]:
print(df.dtypes.value_counts())print()missing = (df.drop(columns=["class_full"]) == "?").sum()print("Cột có giá trị thiếu:")print(missing[missing > 0])print()print("Số giá trị duy nhất mỗi cột:")print(df.nunique())

## Hình 1 — Phân bố biến mục tiêu (class)**Hình cho thấy gì?** Trong 8.124 mẫu, có 4.208 mẫu ăn được (51,8%) và 3.916 mẫu độc (48,2%).**Ý nghĩa với bài toán?** Tỷ lệ giữa 2 lớp gần như cân bằng (51,8% / 48,2%) → đây **không phải** bài toán mất cân bằng dữ liệu (imbalanced), nên Accuracy vẫn là một metric tham khảo hợp lý (dù vẫn nên xem thêm Precision/Recall/F1 vì sai lầm giữa 2 lớp có hậu quả rất khác nhau — ăn nhầm nấm độc nguy hiểm hơn nhiều so với bỏ qua nấm ăn được).**Quyết định xử lý tiếp theo:** Không cần áp dụng kỹ thuật xử lý mất cân bằng (class_weight/resampling) ở bước 2; khi chia train/test vẫn nên dùng `stratify=y` để giữ đúng tỷ lệ.

In [ ]:
plt.figure(figsize=(6,4.5))palette = {"edible (ăn được)": "#4C956C", "poisonous (độc)": "#C1121F"}order = df["class_full"].value_counts().indexax = sns.countplot(data=df, x="class_full", order=order, hue="class_full", palette=palette, legend=False)for p_ in ax.patches:    ax.annotate(f"{int(p_.get_height())}", (p_.get_x()+p_.get_width()/2, p_.get_height()), ha="center", va="bottom")plt.title("Hình 1. Phân bố biến mục tiêu (class)")plt.xlabel(""); plt.ylabel("Số lượng mẫu")plt.tight_layout()plt.savefig("../../docs/figures/fig1_target_distribution.png", dpi=150)plt.show()

## Hình 2 — Bản đồ dữ liệu thiếu**Hình cho thấy gì?** Chỉ duy nhất cột `stalk-root` có giá trị thiếu (ký hiệu `?`), chiếm 30,5% tổng số mẫu (2.480/8.124 dòng). Tất cả 21 cột còn lại không thiếu giá trị nào.**Ý nghĩa với bài toán?** Tỷ lệ thiếu khá lớn (~30%) nhưng chỉ tập trung ở 1 cột duy nhất, không lan rộng — có thể xử lý cục bộ mà không ảnh hưởng các thuộc tính khác.**Quyết định xử lý tiếp theo:** Ở bước 2, coi `?` là một **giá trị hợp lệ riêng** (một mức phân loại "unknown") thay vì xoá dòng hoặc điền mode — vì xoá 30% dữ liệu là quá nhiều, và việc "không xác định được gốc cuống" tự nó cũng có thể là một đặc điểm mang thông tin phân biệt.

In [ ]:
missing_pct = (df.drop(columns=["class_full"]) == "?").sum() / len(df) * 100missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)plt.figure(figsize=(6.5,3.5))bars = plt.barh(missing_pct.index.astype(str), missing_pct.values, color="#E07A5F")for b in bars:    plt.text(b.get_width()+0.3, b.get_y()+b.get_height()/2, f"{b.get_width():.1f}%", va="center")plt.title("Hình 2. Tỷ lệ giá trị thiếu ('?') theo cột")plt.xlabel("% giá trị thiếu")plt.xlim(0, max(missing_pct.values)*1.25)plt.tight_layout()plt.savefig("../../docs/figures/fig2_missing_data.png", dpi=150)plt.show()

## Hình 3 — Mùi (odor) theo nhãn class**Hình cho thấy gì?** `odor='n'` (không mùi) gần như chỉ xuất hiện ở nấm ăn được (96,6% edible); các mùi `f, y, s, p, c, m` gần như 100% là nấm độc; `a` (hạnh nhân) và `l` (cỏ linh lăng) 100% là nấm ăn được.**Ý nghĩa với bài toán?** `odor` là thuộc tính có sức phân biệt mạnh nhất trong toàn bộ 22 thuộc tính (khẳng định lại ở Hình 4) — chỉ riêng mùi đã gần như quyết định được nhãn.**Quyết định xử lý tiếp theo:** Giữ nguyên `odor` làm thuộc tính đầu vào chính; khi mã hoá One-Hot cần đảm bảo không loại bỏ nhầm mức hiếm (`m` chỉ có ~30-40 mẫu) trong lúc chia train/test.

In [ ]:
plt.figure(figsize=(9,5))odor_order = df["odor"].value_counts().indexsns.countplot(data=df, x="odor", hue="class_full", order=odor_order, palette=palette)plt.title("Hình 3. Phân bố mùi (odor) theo nhãn class")plt.xlabel("odor (mã ký tự)"); plt.ylabel("Số lượng mẫu")plt.legend(title="")plt.tight_layout()plt.savefig("../../docs/figures/fig3_odor_vs_class.png", dpi=150)plt.show()print(df.groupby("odor")["class"].value_counts(normalize=True).unstack().fillna(0))

## Hình 4 — Mức độ liên hệ giữa các thuộc tính (Cramér's V)**Hình cho thấy gì?** Vì toàn bộ thuộc tính là categorical nên dùng hệ số **Cramér's V** (0–1) thay cho hệ số tương quan Pearson. `odor` có liên hệ mạnh nhất với `class` (V ≈ 0,97), tiếp theo là `spore-print-color` (0,75), `gill-color` (0,68), `ring-type` (0,60).**Ý nghĩa với bài toán?** Có một nhóm nhỏ thuộc tính mang phần lớn thông tin phân biệt (odor, spore-print-color, gill-color); một số cặp thuộc tính đầu vào cũng liên hệ khá mạnh với nhau (ví dụ `bruises`–`ring-type` ≈ 0,77) → có khả năng dư thừa thông tin (redundancy) giữa các đặc trưng.**Quyết định xử lý tiếp theo:** Không loại bỏ thuộc tính nào ở bước EDA (để các mô hình cây/boosting tự chọn đặc trưng quan trọng), nhưng sẽ **loại `veil-type`** (hằng số, Cramér's V không xác định do chỉ có 1 giá trị) ở bước xử lý dữ liệu vì không mang thông tin.

In [ ]:
def cramers_v(x, y):    confusion = pd.crosstab(x, y)    chi2 = chi2_contingency(confusion)[0]    n = confusion.values.sum()    r, k = confusion.shape    phi2 = chi2 / n    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))    rcorr = r - ((r-1)**2)/(n-1)    kcorr = k - ((k-1)**2)/(n-1)    denom = min((kcorr-1),(rcorr-1))    return np.sqrt(phi2corr/denom) if denom > 0 else 0selected = ["class","odor","gill-color","spore-print-color","ring-type",            "population","habitat","cap-color","bruises","stalk-surface-above-ring"]mat = pd.DataFrame(index=selected, columns=selected, dtype=float)for a, b in combinations(selected, 2):    v = cramers_v(df[a], df[b])    mat.loc[a,b] = v; mat.loc[b,a] = vfor a in selected:    mat.loc[a,a] = 1.0plt.figure(figsize=(8.5,7))sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1, square=True,            cbar_kws={"label":"Cramér's V"})plt.title("Hình 4. Mức độ liên hệ (Cramér's V) giữa các thuộc tính chọn lọc")plt.tight_layout()plt.savefig("../../docs/figures/fig4_cramers_v_heatmap.png", dpi=150)plt.show()

## Hình 5 — Màu bào tử in (spore-print-color) theo nhãn class**Hình cho thấy gì?** `spore-print-color='h'` gần như chỉ xuất hiện ở nấm độc (97,1%); `'w'` cũng thiên về độc (75,9%); trong khi `'b','o','u','y'` gần như 100% ăn được.**Ý nghĩa với bài toán?** Đây là thuộc tính có sức phân biệt mạnh thứ 2 sau `odor` — xác nhận lại kết quả ở Hình 4.**Quyết định xử lý tiếp theo:** Giữ nguyên làm đầu vào; đây là ứng viên tốt để kiểm tra "feature importance" sau khi huấn luyện mô hình cây ở bước 3–4, đối chiếu xem có khớp với phân tích EDA không.

In [ ]:
plt.figure(figsize=(9,5))spc_order = df["spore-print-color"].value_counts().indexsns.countplot(data=df, x="spore-print-color", hue="class_full", order=spc_order, palette=palette)plt.title("Hình 5. Màu bào tử in (spore-print-color) theo nhãn class")plt.xlabel("spore-print-color (mã ký tự)"); plt.ylabel("Số lượng mẫu")plt.legend(title="")plt.tight_layout()plt.savefig("../../docs/figures/fig5_sporeprint_vs_class.png", dpi=150)plt.show()

## Hình 6 — Mật độ quần thể (population) và môi trường sống (habitat) theo class**Hình cho thấy gì?** Nấm mọc đơn lẻ/rải rác (`population='y','v'`) và ở môi trường đường/đô thị, vườn (`habitat='p','u'`) có tỷ lệ độc cao hơn hẳn so với nấm mọc thành cụm đông (`population='a','n'`) ở đồng cỏ (`habitat='g'`).**Ý nghĩa với bài toán?** Hai thuộc tính này có liên hệ vừa phải với nhãn (Cramér's V ≈ 0,49 và 0,44) — không mạnh bằng odor/spore-print-color nhưng vẫn có giá trị bổ sung, đặc biệt trong các trường hợp thiếu thông tin mùi.**Quyết định xử lý tiếp theo:** Giữ nguyên 2 thuộc tính; không cần xử lý đặc biệt (không có giá trị thiếu, số mức không quá nhiều: population có 6 mức, habitat có 7 mức, phù hợp One-Hot Encoding).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))sns.countplot(data=df, x="population", hue="class_full", palette=palette, ax=axes[0],              order=df["population"].value_counts().index)axes[0].set_title("Population theo class"); axes[0].legend(title="", fontsize=8)sns.countplot(data=df, x="habitat", hue="class_full", palette=palette, ax=axes[1],              order=df["habitat"].value_counts().index)axes[1].set_title("Habitat theo class"); axes[1].legend(title="", fontsize=8)plt.suptitle("Hình 6. Population và Habitat theo nhãn class")plt.tight_layout()plt.savefig("../../docs/figures/fig6_population_habitat_vs_class.png", dpi=150)plt.show()

## Tổng kết quyết định xử lý dữ liệu (chuyển sang bước 2 — `02_preprocess`)1. Loại bỏ cột `veil-type` (chỉ có 1 giá trị duy nhất, không mang thông tin).2. Giữ `stalk-root` và coi `'?'` là một mức phân loại riêng (không xoá dòng, không điền mode).3. Mã hoá toàn bộ 21 thuộc tính còn lại bằng One-Hot Encoding (không có thuộc tính dạng số).4. Chia train/test với `stratify=y` (do 2 lớp gần cân bằng nhưng vẫn nên giữ đúng tỷ lệ).5. Không cần kỹ thuật xử lý mất cân bằng dữ liệu (class_weight/SMOTE).